In [ ]:
!pip install -q datasets captum

In [ ]:
!pip install transformers==4.41.1

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from tqdm import tqdm

from captum.attr import LayerIntegratedGradients
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# - "AIRI-Institute/gena-lm-bert-base-t2t" (BERT-style, BPE tokenization)
# - "PoetschLab/GROVER" (BPE tokenization)
# - "InstaDeepAI/nucleotide-transformer-500m-1000g" (k-mer tokenization)

MODEL_NAME = "PoetschLab/GROVER"
POOLING_STRATEGY = 'mean'  # Options: 'mean', 'cls'
CLASSIFIER_EPOCHS = 20
BATCH_SIZE = 16
MAX_SEQ_LENGTH = 512
N_ATTRIBUTION_STEPS = 50
TOP_K_TOKENS = 10 

print(f"Model: {MODEL_NAME}")
print(f"Pooling: {POOLING_STRATEGY}")
print(f"Training epochs: {CLASSIFIER_EPOCHS}")

In [ ]:
dataset = load_dataset("katarinagresova/Genomic_Benchmarks_demo_coding_vs_intergenomic_seqs")

train_dataset = dataset["train"]
test_dataset = dataset["test"]

print(f"\nTraining samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

train_labels = [item['label'] for item in train_dataset]
test_labels = [item['label'] for item in test_dataset]

print(f"\nTraining set - Coding: {train_labels.count(0)}, Non-coding: {train_labels.count(1)}")
print(f"Test set - Coding: {test_labels.count(0)}, Non-coding: {test_labels.count(1)}")

In [ ]:
def load_genome_model(model_name, device):
    print(f"Loading model: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModel.from_pretrained(
        model_name, trust_remote_code=True,
    ).to(device)
    model.eval()
    return model, tokenizer

genome_model, tokenizer = load_genome_model(MODEL_NAME, device)

In [ ]:
def generate_embeddings(model, tokenizer, sequences, pooling='mean', batch_size=20, device='cuda'):
    model.eval()
    all_embeddings = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(sequences), batch_size), desc="Generating embeddings"):
            batch_seqs = sequences[i:i+batch_size]
            
            # Tokenize
            inputs = tokenizer(
                batch_seqs,
                return_tensors="pt",
                padding=True,
                truncation=True,
            ).to(device)
            
            # Forward pass
            outputs = model(**inputs, output_hidden_states=True)
            final_hidden = outputs.hidden_states[-1]  # Final layer
            
            # Pool to get sequence embeddings
            if pooling == 'mean':
                attention_mask = inputs.get('attention_mask')
                if attention_mask is not None:
                    # Masked mean pooling
                    mask_expanded = attention_mask.unsqueeze(-1).expand(final_hidden.size()).float()
                    sum_embeddings = torch.sum(final_hidden * mask_expanded, 1)
                    sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
                    embeddings = sum_embeddings / sum_mask
                else:
                    embeddings = final_hidden.mean(dim=1)
            elif pooling == 'cls':
                embeddings = final_hidden[:, 0, :]  # CLS token
            
            all_embeddings.append(embeddings.cpu().numpy())
    
    return np.vstack(all_embeddings)

train_sequences = [item['seq'] for item in train_dataset]
train_labels_array = np.array([item['label'] for item in train_dataset])
train_embeddings = generate_embeddings(
    genome_model, tokenizer, train_sequences, 
    pooling=POOLING_STRATEGY, batch_size=BATCH_SIZE, device=device
)
print(f"Train embeddings shape: {train_embeddings.shape}")

In [ ]:
class CodingNonCodingClassifier(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=256):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 2)  # Binary: [coding, non-coding]
        )
    
    def forward(self, x):
        return self.classifier(x)

def train_classifier(embeddings, labels, input_dim, epochs=20, device='cuda'):
    classifier = CodingNonCodingClassifier(input_dim=input_dim).to(device)
    optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    X_train = torch.FloatTensor(embeddings).to(device)
    y_train = torch.LongTensor(labels).to(device)
    
    classifier.train()
    print("\nTraining classifier now.")
    
    for epoch in range(epochs):
        # Forward pass
        logits = classifier(X_train)
        loss = criterion(logits, y_train)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Calculate accuracy
        with torch.no_grad():
            predictions = torch.argmax(logits, dim=1)
            accuracy = (predictions == y_train).float().mean().item()
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}")
    
    classifier.eval()
    print("\nTraining completed.")
    return classifier

input_dim = train_embeddings.shape[1]
classifier = train_classifier(
    train_embeddings, 
    train_labels_array, 
    input_dim=input_dim, 
    epochs=CLASSIFIER_EPOCHS, 
    device=device
)

1. test trained classifier
2. attributions only on correct predictions
3. look at incorrect predictions, and attributions in that case
4. look for uniquely top sequences for each type
5. see how frequently tokens occur within each sequence of coding/noncoding test set

In [ ]:
def test_classifier_on_embeddings(classifier, genome_model, tokenizer, test_dataset, pooling='mean', batch_size=16, device='cuda'):
    model = genome_model
    classifier.eval()
    all_predictions = []
    all_true_labels = []
    all_indices = []

    test_sequences = [item['seq'] for item in test_dataset]
    test_labels = [item['label'] for item in test_dataset]

    # Generate test embeddings
    test_embeddings = generate_embeddings(model, tokenizer, test_sequences,
                                         pooling=pooling, batch_size=batch_size,
                                         device=device)
    with torch.no_grad():
        logits = classifier(torch.FloatTensor(test_embeddings).to(device))
        predictions = torch.argmax(logits, dim=1).cpu().numpy()
        accuracy = np.mean(predictions == np.array(test_labels))
        print(f'\nTest set accuracy: {accuracy:.4f}')
        print('Confusion matrix:')
        from sklearn.metrics import confusion_matrix
        print(confusion_matrix(test_labels, predictions))

    return predictions, test_labels, test_sequences

test_predictions, test_labels, test_sequences = test_classifier_on_embeddings(
    classifier, genome_model, tokenizer, test_dataset,
    pooling=POOLING_STRATEGY, batch_size=BATCH_SIZE, device=device
)

In [ ]:
class GenomeLMWithClassifier(nn.Module):
    """Combined model: Genome LM + Classifier for end-to-end attribution"""
    def __init__(self, genome_model, classifier, pooling='mean'):
        super().__init__()
        self.genome_model = genome_model
        self.classifier = classifier
        self.pooling = pooling
    
    def forward(self, input_ids=None, inputs_embeds=None, attention_mask=None):
        """
        Forward pass through entire model
        Supports both input_ids (normal) and inputs_embeds (for LIG)
        """
        # Get outputs from genome model
        if inputs_embeds is not None:
            outputs = self.genome_model(
                inputs_embeds=inputs_embeds,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
        else:
            outputs = self.genome_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
        
        # Get final layer output
        final_hidden = outputs.hidden_states[-1]
        
        # Pool to get sequence embedding
        if self.pooling == 'mean':
            if attention_mask is not None:
                mask_expanded = attention_mask.unsqueeze(-1).expand(final_hidden.size()).float()
                sum_embeddings = torch.sum(final_hidden * mask_expanded, 1)
                sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
                sequence_embedding = sum_embeddings / sum_mask
            else:
                sequence_embedding = final_hidden.mean(dim=1)
        elif self.pooling == 'cls':
            sequence_embedding = final_hidden[:, 0, :]
        
        # Classify
        logits = self.classifier(sequence_embedding)
        
        # Return logit for coding class (index 0)
        return logits[:, 0]

combined_model = GenomeLMWithClassifier(genome_model, classifier, pooling=POOLING_STRATEGY)
combined_model.eval()
print("Combined model created!")

In [ ]:
def get_token_attributions(combined_model, tokenizer, sequence, device='cuda', n_steps=50):
    combined_model.eval()
    
    # Tokenize
    inputs = tokenizer(
        sequence,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH
    ).to(device)
    
    input_ids = inputs['input_ids']
    attention_mask = inputs.get('attention_mask')
    
    # Get embedding layer
    try:
        embedding_layer = combined_model.genome_model.embeddings.word_embeddings
    except AttributeError:
        try:
            embedding_layer = combined_model.genome_model.model.embeddings.word_embeddings
        except AttributeError:
            embedding_layer = combined_model.genome_model.embeddings
    
    # Initialize LayerIntegratedGradients
    lig = LayerIntegratedGradients(combined_model, embedding_layer)
    
    # Baseline: zero embeddings
    baseline_ids = torch.zeros_like(input_ids)
    print('starting attribution')
    # Compute attributions
    try:
        if attention_mask is not None:
            attributions = lig.attribute(
                inputs=input_ids,
                baselines=baseline_ids,
                n_steps=n_steps,
                internal_batch_size=1
            )
        else:
            attributions = lig.attribute(
                inputs=input_ids,
                baselines=baseline_ids,
                n_steps=n_steps,
                internal_batch_size=1
            )
    except Exception as e:
        print(f"Attribution error: {e}")
        return None

    print('computed attributions')
    
    # Sum across embedding dimension to get per-token attribution
    token_attributions = attributions.sum(dim=-1).squeeze(0).cpu().detach().numpy()
    
    # Get tokens
    tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze().tolist())
    
    # Filter special tokens and get positions
    filtered_data = []
    token_position = 0
    
    for idx, (tok, attr) in enumerate(zip(tokens, token_attributions)):
        if tok not in ['[CLS]', '[SEP]', '<s>', '</s>', '<pad>', '[PAD]', '<unk>', '[UNK]']:
            filtered_data.append({
                'token': tok,
                'attribution': float(attr),
                'token_id': idx,
                'position': token_position
            })
            token_position += 1
    
    return filtered_data

def find_token_positions_in_sequence(token, sequence, tokenizer):
    """
    Find approximate positions of a token in the original DNA sequence
    
    Args:
        token: Token string
        sequence: Original DNA sequence
        tokenizer: Tokenizer
    
    Returns:
        List of positions where token appears
    """
    # Remove special characters from token
    clean_token = token.replace('▁', '').replace('##', '').replace('Ġ', '')
    
    # Find all occurrences in sequence
    positions = []
    start = 0
    while True:
        pos = sequence.find(clean_token, start)
        if pos == -1:
            break
        positions.append(pos)
        start = pos + 1
    
    return positions if positions else [-1]

In [ ]:
"""
test_samples = []
codingCount, noncodingCount = 0, 0
for x in test_dataset:
    if x['label'] == 0 and codingCount <= 250:
        codingCount += 1
        test_samples.append(x)
    if x['label'] == 1 and noncodingCount <= 250:
        noncodingCount += 1
        test_samples.append(x)
    if codingCount == noncodingCount == 250:
        break
"""

In [ ]:
def analyze_test_set_separate_correct_incorrect(
        combined_model, tokenizer, test_dataset, test_predictions,
        test_labels, device='cuda', n_steps=50, top_k=10, max_samples=None
    ):
    """
    Run attribution analysis on test set, record for correct/incorrect predictions separately.
    """
    results_correct = []
    results_incorrect = []
    token_frequency_correct = Counter()
    token_frequency_incorrect = Counter()

    n_samples = len(test_dataset) if max_samples is None else min(max_samples, len(test_dataset))
    print(f"\nAnalyzing {n_samples} test sequences (segregating correct vs incorrect predictions)...\n")

    for idx in tqdm(range(n_samples)):
        sequence = test_dataset[idx]['seq']
        true_label = test_labels[idx]
        pred_label = test_predictions[idx]
        label_name = 'coding' if true_label == 0 else 'non-coding'

        # Attribution
        attribution_data = get_token_attributions(
            combined_model, tokenizer, sequence, device=device, n_steps=n_steps
        )

        if attribution_data is None:
            continue

        sorted_data = sorted(attribution_data, key=lambda x: abs(x['attribution']), reverse=True)
        top_tokens = sorted_data[:top_k]

        result = {
            'sample_id': idx,
            'sequence_type': label_name,
            'sequence_length': len(sequence),
            'top_tokens': [t['token'] for t in top_tokens],
            'top_attributions': [t['attribution'] for t in top_tokens],
            'top_token_ids': [t['token_id'] for t in top_tokens],
            'top_token_positions': [t['position'] for t in top_tokens],
            'true_label': true_label,
            'pred_label': pred_label,
        }

        if true_label == pred_label:
            results_correct.append(result)
            for token_info in top_tokens:
                token_frequency_correct[token_info['token']] += 1
        else:
            results_incorrect.append(result)
            for token_info in top_tokens:
                token_frequency_incorrect[token_info['token']] += 1

    # Convert lists to DataFrames
    correct_df = pd.DataFrame(results_correct)
    incorrect_df = pd.DataFrame(results_incorrect)
    return correct_df, incorrect_df, token_frequency_correct, token_frequency_incorrect

# ---- Run this after test_classifier_on_embeddings ----
correct_df, incorrect_df, freq_correct, freq_incorrect = analyze_test_set_separate_correct_incorrect(
    combined_model, tokenizer, test_dataset, test_predictions,
    test_labels, device=device, n_steps=N_ATTRIBUTION_STEPS,
    top_k=TOP_K_TOKENS, max_samples=None # or None for full set
)

print("\nAnalysis complete.")
print(f"Sequences with correct predictions: {len(correct_df)}")
print(f"Sequences with incorrect predictions: {len(incorrect_df)}")


In [ ]:
def get_unique_tokens(freq_coding, freq_noncoding, top_n=10):
    """
    Find tokens that are unique or highly specific to coding or non-coding sequences.
    Returns top N tokens with highest frequency difference.
    """
    all_tokens = set(freq_coding.keys()) | set(freq_noncoding.keys())
    
    coding_unique = []
    noncoding_unique = []
    
    for token in all_tokens:
        coding_count = freq_coding.get(token, 0)
        noncoding_count = freq_noncoding.get(token, 0)
        
        # Calculate difference (positive = coding-specific, negative = noncoding-specific)
        diff = coding_count - noncoding_count
        
        if diff > 0:
            coding_unique.append((token, coding_count, noncoding_count, diff))
        elif diff < 0:
            noncoding_unique.append((token, noncoding_count, coding_count, abs(diff)))
    
    # Sort by difference (descending)
    coding_unique_sorted = sorted(coding_unique, key=lambda x: x[3], reverse=True)[:top_n]
    noncoding_unique_sorted = sorted(noncoding_unique, key=lambda x: x[3], reverse=True)[:top_n]
    
    return coding_unique_sorted, noncoding_unique_sorted

In [ ]:
def display_comprehensive_results(correct_df, incorrect_df, freq_correct, freq_incorrect):
    """
    Display comprehensive results including:
    - Overall statistics
    - Sample results for correct/incorrect predictions
    - Most frequent tokens (overall and by type)
    - Unique tokens for coding vs non-coding
    """
    
    print("="*80)
    print("COMPREHENSIVE ATTRIBUTION ANALYSIS RESULTS")
    print("="*80)
    
    # ========== SECTION 1: Overall Statistics ==========
    print("\n" + "="*80)
    print("1. OVERALL STATISTICS")
    print("="*80)
    
    total_correct = len(correct_df)
    total_incorrect = len(incorrect_df)
    total_analyzed = total_correct + total_incorrect
    
    print(f"\nTotal sequences analyzed: {total_analyzed}")
    print(f"  - Correctly classified: {total_correct} ({100*total_correct/total_analyzed:.2f}%)")
    print(f"  - Incorrectly classified: {total_incorrect} ({100*total_incorrect/total_analyzed:.2f}%)")
    
    if len(correct_df) > 0:
        coding_correct = len(correct_df[correct_df['sequence_type'] == 'coding'])
        noncoding_correct = len(correct_df[correct_df['sequence_type'] == 'non-coding'])
        print(f"\nCorrect predictions breakdown:")
        print(f"  - Coding: {coding_correct}")
        print(f"  - Non-coding: {noncoding_correct}")
    
    if len(incorrect_df) > 0:
        coding_incorrect = len(incorrect_df[incorrect_df['sequence_type'] == 'coding'])
        noncoding_incorrect = len(incorrect_df[incorrect_df['sequence_type'] == 'non-coding'])
        print(f"\nIncorrect predictions breakdown:")
        print(f"  - Coding (misclassified): {coding_incorrect}")
        print(f"  - Non-coding (misclassified): {noncoding_incorrect}")
    
    # ========== SECTION 2: Sample Results (Correct) ==========
    print("\n" + "="*80)
    print("2. SAMPLE RESULTS - CORRECT PREDICTIONS (First 3)")
    print("="*80)
    
    for idx in range(min(3, len(correct_df))):
        row = correct_df.iloc[idx]
        print(f"\n{'-'*60}")
        print(f"Sequence {row['sample_id']} - TRUE: {row['sequence_type'].upper()} | PRED: {['coding', 'non-coding'][row['pred_label']].upper()} ✓")
        print(f"Length: {row['sequence_length']} bp")
        print(f"\nTop {TOP_K_TOKENS} Important Tokens:")
        for i, (token, attr, pos) in enumerate(zip(
            row['top_tokens'], row['top_attributions'], row['top_token_positions']
        ), 1):
            print(f"  {i:2d}. Token: {token:15s} | Attribution: {attr:+.4f} | Position: {pos}")
    
    # ========== SECTION 3: Sample Results (Incorrect) ==========
    if len(incorrect_df) > 0:
        print("\n" + "="*80)
        print("3. SAMPLE RESULTS - INCORRECT PREDICTIONS (First 3)")
        print("="*80)
        
        for idx in range(min(3, len(incorrect_df))):
            row = incorrect_df.iloc[idx]
            print(f"\n{'-'*60}")
            print(f"Sequence {row['sample_id']} - TRUE: {row['sequence_type'].upper()} | PRED: {['coding', 'non-coding'][row['pred_label']].upper()} ✗")
            print(f"Length: {row['sequence_length']} bp")
            print(f"\nTop {TOP_K_TOKENS} Important Tokens:")
            for i, (token, attr, pos) in enumerate(zip(
                row['top_tokens'], row['top_attributions'], row['top_token_positions']
            ), 1):
                print(f"  {i:2d}. Token: {token:15s} | Attribution: {attr:+.4f} | Position: {pos}")
    
    # ========== SECTION 4: Most Frequent Tokens (Correct) ==========
    print("\n" + "="*80)
    print("4. MOST FREQUENT IMPORTANT TOKENS - CORRECT PREDICTIONS")
    print("="*80)
    
    most_common_correct = freq_correct.most_common(20)
    print(f"\nTop 20 most frequently appearing important tokens (correct predictions):")
    print(f"\n{'Rank':<6} {'Token':<20} {'Frequency':<12}")
    print("-" * 40)
    for rank, (token, count) in enumerate(most_common_correct, 1):
        print(f"{rank:<6} {token:<20} {count:<12}")
    
    # ========== SECTION 5: Most Frequent Tokens (Incorrect) ==========
    if len(incorrect_df) > 0:
        print("\n" + "="*80)
        print("5. MOST FREQUENT IMPORTANT TOKENS - INCORRECT PREDICTIONS")
        print("="*80)
        
        most_common_incorrect = freq_incorrect.most_common(20)
        print(f"\nTop 20 most frequently appearing important tokens (incorrect predictions):")
        print(f"\n{'Rank':<6} {'Token':<20} {'Frequency':<12}")
        print("-" * 40)
        for rank, (token, count) in enumerate(most_common_incorrect, 1):
            print(f"{rank:<6} {token:<20} {count:<12}")
    
    # ========== SECTION 6: Separate by Sequence Type (Correct) ==========
    print("\n" + "="*80)
    print("6. TOKEN FREQUENCY BY SEQUENCE TYPE - CORRECT PREDICTIONS")
    print("="*80)
    
    if len(correct_df) > 0:
        # Separate by type
        coding_correct_df = correct_df[correct_df['sequence_type'] == 'coding']
        noncoding_correct_df = correct_df[correct_df['sequence_type'] == 'non-coding']
        
        # Get tokens for each type
        coding_tokens = []
        for tokens in coding_correct_df['top_tokens']:
            coding_tokens.extend(tokens)
        
        noncoding_tokens = []
        for tokens in noncoding_correct_df['top_tokens']:
            noncoding_tokens.extend(tokens)
        
        freq_coding = Counter(coding_tokens)
        freq_noncoding = Counter(noncoding_tokens)
        
        print(f"\n--- CODING Sequences (n={len(coding_correct_df)}) ---")
        print(f"{'Rank':<6} {'Token':<20} {'Frequency':<12}")
        print("-" * 40)
        for rank, (token, count) in enumerate(freq_coding.most_common(15), 1):
            print(f"{rank:<6} {token:<20} {count:<12}")
        
        print(f"\n--- NON-CODING Sequences (n={len(noncoding_correct_df)}) ---")
        print(f"{'Rank':<6} {'Token':<20} {'Frequency':<12}")
        print("-" * 40)
        for rank, (token, count) in enumerate(freq_noncoding.most_common(15), 1):
            print(f"{rank:<6} {token:<20} {count:<12}")
        
        # ========== SECTION 7: Unique Tokens Analysis ==========
        print("\n" + "="*80)
        print("7. UNIQUE TOKENS ANALYSIS - CORRECT PREDICTIONS")
        print("="*80)
        
        coding_unique, noncoding_unique = get_unique_tokens(freq_coding, freq_noncoding, top_n=10)
        
        print("\n--- Top 10 CODING-SPECIFIC Tokens ---")
        print("(Tokens most frequently important for coding vs non-coding)")
        print(f"\n{'Rank':<6} {'Token':<15} {'Coding':<10} {'Non-coding':<12} {'Difference':<12}")
        print("-" * 60)
        for rank, (token, coding_count, noncoding_count, diff) in enumerate(coding_unique, 1):
            print(f"{rank:<6} {token:<15} {coding_count:<10} {noncoding_count:<12} +{diff:<11}")
        
        print("\n--- Top 10 NON-CODING-SPECIFIC Tokens ---")
        print("(Tokens most frequently important for non-coding vs coding)")
        print(f"\n{'Rank':<6} {'Token':<15} {'Non-coding':<12} {'Coding':<10} {'Difference':<12}")
        print("-" * 60)
        for rank, (token, noncoding_count, coding_count, diff) in enumerate(noncoding_unique, 1):
            print(f"{rank:<6} {token:<15} {noncoding_count:<12} {coding_count:<10} +{diff:<11}")
    
    # ========== SECTION 8: Incorrect Predictions Analysis ==========
    if len(incorrect_df) > 0:
        print("\n" + "="*80)
        print("8. TOKEN FREQUENCY BY SEQUENCE TYPE - INCORRECT PREDICTIONS")
        print("="*80)
        
        coding_incorrect_df = incorrect_df[incorrect_df['sequence_type'] == 'coding']
        noncoding_incorrect_df = incorrect_df[incorrect_df['sequence_type'] == 'non-coding']
        
        # Get tokens for misclassified sequences
        coding_incorrect_tokens = []
        for tokens in coding_incorrect_df['top_tokens']:
            coding_incorrect_tokens.extend(tokens)
        
        noncoding_incorrect_tokens = []
        for tokens in noncoding_incorrect_df['top_tokens']:
            noncoding_incorrect_tokens.extend(tokens)
        
        freq_coding_incorrect = Counter(coding_incorrect_tokens)
        freq_noncoding_incorrect = Counter(noncoding_incorrect_tokens)
        
        if len(coding_incorrect_df) > 0:
            print(f"\n--- CODING Sequences (MISCLASSIFIED, n={len(coding_incorrect_df)}) ---")
            print(f"{'Rank':<6} {'Token':<20} {'Frequency':<12}")
            print("-" * 40)
            for rank, (token, count) in enumerate(freq_coding_incorrect.most_common(15), 1):
                print(f"{rank:<6} {token:<20} {count:<12}")
        
        if len(noncoding_incorrect_df) > 0:
            print(f"\n--- NON-CODING Sequences (MISCLASSIFIED, n={len(noncoding_incorrect_df)}) ---")
            print(f"{'Rank':<6} {'Token':<20} {'Frequency':<12}")
            print("-" * 40)
            for rank, (token, count) in enumerate(freq_noncoding_incorrect.most_common(15), 1):
                print(f"{rank:<6} {token:<20} {count:<12}")


# ========== Cell: Run Comprehensive Analysis ==========
display_comprehensive_results(correct_df, incorrect_df, freq_correct, freq_incorrect)

In [ ]:
def save_enhanced_results(correct_df, incorrect_df, freq_correct, freq_incorrect):
    """
    Save all results to CSV files
    """
    model_short_name = MODEL_NAME.split('/')[-1]
    
    # 1. Save correct predictions
    if len(correct_df) > 0:
        correct_export = correct_df.copy()
        correct_export['top_tokens'] = correct_export['top_tokens'].apply(lambda x: '|'.join(x))
        correct_export['top_attributions'] = correct_export['top_attributions'].apply(
            lambda x: '|'.join([f"{a:.4f}" for a in x])
        )
        correct_export['top_token_ids'] = correct_export['top_token_ids'].apply(
            lambda x: '|'.join([str(i) for i in x])
        )
        correct_export['top_token_positions'] = correct_export['top_token_positions'].apply(
            lambda x: '|'.join([str(i) for i in x])
        )
        
        correct_filename = f"attribution_results_CORRECT_{model_short_name}.csv"
        correct_export.to_csv(correct_filename, index=False)
        print(f"Correct predictions saved to: {correct_filename}")
    
    # 2. Save incorrect predictions
    if len(incorrect_df) > 0:
        incorrect_export = incorrect_df.copy()
        incorrect_export['top_tokens'] = incorrect_export['top_tokens'].apply(lambda x: '|'.join(x))
        incorrect_export['top_attributions'] = incorrect_export['top_attributions'].apply(
            lambda x: '|'.join([f"{a:.4f}" for a in x])
        )
        incorrect_export['top_token_ids'] = incorrect_export['top_token_ids'].apply(
            lambda x: '|'.join([str(i) for i in x])
        )
        incorrect_export['top_token_positions'] = incorrect_export['top_token_positions'].apply(
            lambda x: '|'.join([str(i) for i in x])
        )
        
        incorrect_filename = f"attribution_results_INCORRECT_{model_short_name}.csv"
        incorrect_export.to_csv(incorrect_filename, index=False)
        print(f"Incorrect predictions saved to: {incorrect_filename}")
    
    # 3. Save token frequencies
    freq_correct_df = pd.DataFrame([
        {'token': token, 'frequency': count, 'prediction': 'correct'}
        for token, count in freq_correct.most_common()
    ])
    
    freq_incorrect_df = pd.DataFrame([
        {'token': token, 'frequency': count, 'prediction': 'incorrect'}
        for token, count in freq_incorrect.most_common()
    ])
    
    freq_combined = pd.concat([freq_correct_df, freq_incorrect_df], ignore_index=True)
    freq_filename = f"token_frequency_detailed_{model_short_name}.csv"
    freq_combined.to_csv(freq_filename, index=False)
    print(f"Token frequencies saved to: {freq_filename}")
    
    # 4. Save unique tokens analysis
    if len(correct_df) > 0:
        coding_df = correct_df[correct_df['sequence_type'] == 'coding']
        noncoding_df = correct_df[correct_df['sequence_type'] == 'non-coding']
        
        coding_tokens = []
        for tokens in coding_df['top_tokens']:
            coding_tokens.extend(tokens)
        
        noncoding_tokens = []
        for tokens in noncoding_df['top_tokens']:
            noncoding_tokens.extend(tokens)
        
        freq_coding = Counter(coding_tokens)
        freq_noncoding = Counter(noncoding_tokens)
        
        coding_unique, noncoding_unique = get_unique_tokens(freq_coding, freq_noncoding, top_n=20)
        
        unique_tokens_df = pd.DataFrame(
            [{'token': t[0], 'type': 'coding-specific', 'type_freq': t[1], 
              'other_freq': t[2], 'difference': t[3]} for t in coding_unique] +
            [{'token': t[0], 'type': 'non-coding-specific', 'type_freq': t[1], 
              'other_freq': t[2], 'difference': t[3]} for t in noncoding_unique]
        )
        
        unique_filename = f"unique_tokens_{model_short_name}.csv"
        unique_tokens_df.to_csv(unique_filename, index=False)
        print(f"Unique tokens analysis saved to: {unique_filename}")

# Save all results
save_enhanced_results(correct_df, incorrect_df, freq_correct, freq_incorrect)

In [ ]:
def visualize_unique_tokens(correct_df):
    """
    Visualize coding-specific vs non-coding-specific tokens
    """
    if len(correct_df) == 0:
        print("No correct predictions to visualize")
        return
    
    # Separate by type
    coding_df = correct_df[correct_df['sequence_type'] == 'coding']
    noncoding_df = correct_df[correct_df['sequence_type'] == 'non-coding']
    
    # Get token frequencies
    coding_tokens = []
    for tokens in coding_df['top_tokens']:
        coding_tokens.extend(tokens)
    
    noncoding_tokens = []
    for tokens in noncoding_df['top_tokens']:
        noncoding_tokens.extend(tokens)
    
    freq_coding = Counter(coding_tokens)
    freq_noncoding = Counter(noncoding_tokens)
    
    # Get unique tokens
    coding_unique, noncoding_unique = get_unique_tokens(freq_coding, freq_noncoding, top_n=10)
    
    # Create comparison plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Coding-specific tokens
    if coding_unique:
        tokens_coding = [t[0] for t in coding_unique]
        diffs_coding = [t[3] for t in coding_unique]
        
        ax1.barh(range(len(tokens_coding)), diffs_coding, color='steelblue')
        ax1.set_yticks(range(len(tokens_coding)))
        ax1.set_yticklabels(tokens_coding)
        ax1.set_xlabel('Frequency Difference (Coding - Non-coding)', fontsize=11)
        ax1.set_title('Top 10 Coding-Specific Tokens', fontsize=13, fontweight='bold')
        ax1.invert_yaxis()
    
    # Non-coding-specific tokens
    if noncoding_unique:
        tokens_noncoding = [t[0] for t in noncoding_unique]
        diffs_noncoding = [t[3] for t in noncoding_unique]
        
        ax2.barh(range(len(tokens_noncoding)), diffs_noncoding, color='coral')
        ax2.set_yticks(range(len(tokens_noncoding)))
        ax2.set_yticklabels(tokens_noncoding)
        ax2.set_xlabel('Frequency Difference (Non-coding - Coding)', fontsize=11)
        ax2.set_title('Top 10 Non-Coding-Specific Tokens', fontsize=13, fontweight='bold')
        ax2.invert_yaxis()
    
    plt.tight_layout()
    plt.show()

# Run visualization
visualize_unique_tokens(correct_df)

# Old code:

In [ ]:
def analyze_test_set(combined_model, tokenizer, test_dataset, device='cuda', 
                     n_steps=50, top_k=10, max_samples=None):
    """
    Run attribution analysis on test set
    
    Args:
        combined_model: Combined model
        tokenizer: Tokenizer
        test_dataset: Test dataset
        device: Device
        n_steps: Integration steps
        top_k: Top k important tokens per sequence
        max_samples: Maximum number of samples to analyze (None = all)
    
    Returns:
        results_df: DataFrame with results
        token_frequency: Counter of important tokens across all sequences
    """
    results = []
    all_important_tokens = []
    
    # Determine how many samples to process
    n_samples = len(test_dataset) if max_samples is None else min(max_samples, len(test_dataset))
    
    print(f"\nAnalyzing {n_samples} test sequences.")
    
    for idx in tqdm(range(n_samples)):
        sample = test_dataset[idx]
        sequence = sample['seq']
        true_label = sample['label']
        label_name = 'coding' if true_label == 0 else 'non-coding'
        
        # Get attributions
        attribution_data = get_token_attributions(
            combined_model, tokenizer, sequence, device=device, n_steps=n_steps
        )
        
        if attribution_data is None:
            continue
        
        # Sort by attribution magnitude (absolute value)
        sorted_data = sorted(attribution_data, key=lambda x: abs(x['attribution']), reverse=True)
        
        # Get top k tokens
        top_tokens = sorted_data[:top_k]
        
        # Record important tokens for frequency analysis
        for token_info in top_tokens:
            all_important_tokens.append(token_info['token'])
        
        # Prepare result entry
        result = {
            'sample_id': idx,
            'sequence_type': label_name,
            'sequence_length': len(sequence),
            'top_tokens': [t['token'] for t in top_tokens],
            'top_attributions': [t['attribution'] for t in top_tokens],
            'top_token_ids': [t['token_id'] for t in top_tokens],
            'top_token_positions': [t['position'] for t in top_tokens]
        }
        
        results.append(result)
    
    # Create DataFrame
    results_df = pd.DataFrame(results)
    
    # Count token frequency
    token_frequency = Counter(all_important_tokens)
    
    return results_df, token_frequency

# Run analysis
# For demonstration, analyze first 100 samples (change to None to analyze all 25000)
"""
results_df, token_frequency = analyze_test_set(
    combined_model, 
    tokenizer, 
    test_samples, 
    device=device,
    n_steps=N_ATTRIBUTION_STEPS,
    top_k=TOP_K_TOKENS,
    max_samples=500  # Change to None for full test set (will take hours!)
)

print(f"\nAnalysis complete! Processed {len(results_df)} sequences.")
"""

In [ ]:
# Display summary
print("="*70)
print("SUMMARY STATISTICS")
print("="*70)

print(f"\nTotal sequences analyzed: {len(results_df)}")
print(f"Coding sequences: {len(results_df[results_df['sequence_type'] == 'coding'])}")
print(f"Non-coding sequences: {len(results_df[results_df['sequence_type'] == 'non-coding'])}")

# Show first few results
print("\n" + "="*70)
print("SAMPLE RESULTS (First 5 sequences)")
print("="*70)

for idx in range(min(10, len(results_df))):
    row = results_df.iloc[idx]
    print(f"\nSequence {row['sample_id']} ({row['sequence_type'].upper()})")
    print("-" * 50)
    print(f"Sequence length: {row['sequence_length']} bp")
    print(f"\nTop {TOP_K_TOKENS} Important Tokens:")
    for i, (token, attr, pos) in enumerate(zip(
        row['top_tokens'], 
        row['top_attributions'], 
        row['top_token_positions']
    ), 1):
        print(f"  {i}. Token: {token:15s} | Attribution: {attr:+.4f} | Position: {pos}")

In [ ]:
# Separate by sequence type
coding_results = results_df[results_df['sequence_type'] == 'coding']
noncoding_results = results_df[results_df['sequence_type'] == 'non-coding']

# Get tokens for each type
coding_tokens = []
for tokens in coding_results['top_tokens']:
    coding_tokens.extend(tokens)

noncoding_tokens = []
for tokens in noncoding_results['top_tokens']:
    noncoding_tokens.extend(tokens)

coding_token_freq = Counter(coding_tokens)
noncoding_token_freq = Counter(noncoding_tokens)

print("\n" + "="*70)
print("COMPARISON: CODING vs NON-CODING SEQUENCES")
print("="*70)

print(f"\nTop 15 tokens for CODING sequences:")
for rank, (token, count) in enumerate(coding_token_freq.most_common(15), 1):
    print(f"  {rank}. {token:20s} (n={count})")

print(f"\nTop 15 tokens for NON-CODING sequences:")
for rank, (token, count) in enumerate(noncoding_token_freq.most_common(15), 1):
    print(f"  {rank}. {token:20s} (n={count})")

In [ ]:
# Save results to CSV
model_short_name = MODEL_NAME.split('/')[-1]
output_filename = f"attribution_results_{model_short_name}.csv"

# Expand lists into strings for CSV
results_df_export = results_df.copy()
results_df_export['top_tokens'] = results_df_export['top_tokens'].apply(lambda x: '|'.join(x))
results_df_export['top_attributions'] = results_df_export['top_attributions'].apply(
    lambda x: '|'.join([f"{a:.4f}" for a in x])
)
results_df_export['top_token_ids'] = results_df_export['top_token_ids'].apply(
    lambda x: '|'.join([str(i) for i in x])
)
results_df_export['top_token_positions'] = results_df_export['top_token_positions'].apply(
    lambda x: '|'.join([str(i) for i in x])
)

results_df_export.to_csv(output_filename, index=False)
print(f"\nResults saved to: {output_filename}")

# Save token frequency
token_freq_df = pd.DataFrame([
    {'token': token, 'frequency': count, 'type': 'all'}
    for token, count in token_frequency.most_common()
])

token_freq_filename = f"token_frequency_{model_short_name}.csv"
token_freq_df.to_csv(token_freq_filename, index=False)
print(f"Token frequencies saved to: {token_freq_filename}")

### Understanding the Results:

1. **Attribution Scores**:
   - **Positive values**: Token increases probability of being "coding"
   - **Negative values**: Token decreases probability of being "coding" (increases "non-coding")
   - **Magnitude**: Larger absolute value = stronger influence

2. **Top Tokens**:
   - These are the most influential tokens for the classification decision
   - Look for biological patterns (e.g., start codons, TATA boxes, splice sites)

3. **Frequent Tokens**:
   - Tokens appearing frequently across sequences suggest general patterns
   - Different frequencies in coding vs non-coding reveal learned distinctions

4. **Next Steps**:
   - Map tokens back to known biological features
   - Check if high-attribution tokens align with regulatory elements
   - Compare across different genome language models
   - Analyze longer sequences or full test set (change `max_samples=None`)

In [ ]:
def getLastLayer(modelName):
    torch.cuda.empty_cache()
    tokenizer = AutoTokenizer.from_pretrained(modelName, trust_remote_code=True)

    # Load Model
    config = BertConfig.from_pretrained("zhihan1996/DNABERT-2-117M")
    model = AutoModel.from_pretrained(
        modelName,
        trust_remote_code=True,
        config=config if modelName == "zhihan1996/DNABERT-2-117M" else None,
    ).to(device)
    model.eval()

    # Target transformer layer (final encoder layer)
    #target_layer = model.encoder.layer[-1]
    target_layer = list(model.children())[-1]  if modelName == "LongSafari/hyenadna-tiny-1k-seqlen-hf" else model.encoder.layer[-1]
    print(f"Target Layer Type for {modelName}:", type(target_layer))